In [30]:
import pandas as pd

df = pd.read_csv("bridge_data.csv")
X = df.drop("Bridge_Condition", axis=1)
y = df["Bridge_Condition"]

In [31]:
num_cols = ["Age_of_Bridge", "Traffic_Volume"]
cat_cols = ["Material_Type", "Maintenance_Level"]

In [32]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [33]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ))
])

model.fit(X_train, y_train)

In [35]:
y_pred = model.predict(X_test)

In [36]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8680555555555556

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.92      0.92       124
           1       0.52      0.55      0.54        20

    accuracy                           0.87       144
   macro avg       0.73      0.73      0.73       144
weighted avg       0.87      0.87      0.87       144


Confusion Matrix:
 [[114  10]
 [  9  11]]


In [40]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__n_estimators": [200, 400],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2],
    "classifier__class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best Parameters: {'classifier__class_weight': 'balanced', 'classifier__max_depth': None, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}
Best CV Score: 0.8993103448275861


In [39]:
from sklearn.metrics import accuracy_score, classification_report

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Test Accuracy: 0.875

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.90      0.93       124
           1       0.54      0.75      0.62        20

    accuracy                           0.88       144
   macro avg       0.75      0.82      0.78       144
weighted avg       0.90      0.88      0.88       144



In [41]:
import joblib

joblib.dump(grid_search.best_estimator_, "best_model.pkl")

['best_model.pkl']

In [42]:
from google.colab import files
files.download("best_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>